# I.26 两角一边：精确有限核验
本 Notebook 是原创教学坐标模型，不是古希腊原证，也不是一般定理的形式化证明。角键用内积除以行列式绝对值；无浮点误差容限。发布记录通过标准库 compile/exec 顺序执行，未通过 Jupyter kernel。请在本目录运行。

In [1]:
from geometry import make_certificate, verify_certificate, triangle, angle_key, length2
from cases import example, positive_cases, negative_cases
print("精确有理坐标；不调用浮点三角函数")

精确有理坐标；不调用浮点三角函数


## 1 夹边案
B 对 E、C 对 F；给定 BC=EF。读取角键和三边平方，直接查看结论。

In [2]:
data = example("included")
certificate = make_certificate(data)
print(certificate["angle_keys"])
print(certificate["squared_sides_AB_BC_AC"])
print(verify_certificate(data, certificate))

[['3/4', '0', '4/3'], ['3/4', '0', '4/3']]
[['9', '16', '25'], ['9', '16', '25']]
True


## 2 对边案
相同角对应，但给定边改为 AB=DE。不能把一般 SSA 误称本命题。

In [3]:
opposite = example("opposite")
print(make_certificate(opposite)["given_edge_index"])
print(verify_certificate(opposite, make_certificate(opposite)))

0
True


## 3 有限覆盖
36 种非退化形状、3 种有理等距变换、2 个给定边分支。包含镜像、钝角与非整数坐标。216 个验收不替代一般证明。

In [4]:
inputs = list(positive_cases())
accepted = sum(verify_certificate(d, make_certificate(d)) for d in inputs)
print({"cases": len(inputs), "accepted": accepted})

{'cases': 216, 'accepted': 216}


## 4 不满足前提的输入
两角相等但缩放、错误顶点对应、共线退化、SSA 均不能当作 I.26 的完整前提。

In [5]:
for name, bad in negative_cases().items():
    try:
        make_certificate(bad)
        raise RuntimeError("unexpected acceptance")
    except ValueError as error:
        print(name, ":", str(error))

angles_without_equal_side : specified corresponding side must agree
wrong_correspondence : corresponding B/E and C/F angles must agree
degenerate : degenerate triangle
ssa_not_two_angles : corresponding B/E and C/F angles must agree


## 5 证书必须绑定调用者的输入
整数 schema=1 不允许被布尔 True 替换；另一合法输入的证书也不允许移植。

In [6]:
import copy
forged = copy.deepcopy(certificate)
forged["schema"] = True
print("bool schema accepted:", verify_certificate(data, forged))
other = example(); other["triangle_2"] = [["20","3"],["20","0"],["16","0"]]
print("different input accepted:", verify_certificate(data, make_certificate(other)))

bool schema accepted: False
different input accepted: False


## 复盘
为什么拒绝共线？行列式为零，角键无定义且三角形假设失效。为什么不直接用 assert？优化模式可删除断言，生产验收用显式异常和布尔返回。为什么不声称原证实跑？本程序直接计算前提和结论，未编码 I.4/I.16 的通用证明规则或几何位置公理。